In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

FLOWER_TYPES = [
    ("Роза",          ["роза", "розы", "розов", "розой", "роз ", "rose", "roses"]),
    ("Тюльпан",       ["тюльпан", "tulip"]),
    ("Гербера",       ["гербер", "герберы", "gerbera", "germini", "гермини"]),
    ("Альстромерия",  ["альстромери", "alstromer"]),
    ("Хризантема",    ["хризантем", "chrys", "сантини", "santini"]),
    ("Дендробиум",    ["дендробиум", "dendrobium"]),
    ("Гвоздика",      ["гвоздик", "dianthus", "диантус"]),
    ("Ирис",          ["ирис", "iris"]),
    ("Пион",          ["пион", "peony", "пеони"]),
    ("Лилия",         ["лили", "lili", "lily"]),
    ("Гортензия",     ["гортензи", "hydrangea"]),
    ("Эустома",       ["эустом", "eustom", "lisianthus", "лизиантус"]),
    ("Подсолнух",     ["подсолнух", "sunflower"]),
    ("Ромашка",       ["ромашк", "chamomile"]),
    ("Сирень",        ["сирен", "lilac"]),
    ("Орхидея",       ["орхидея", "орхидей", "orchid"]),
    ("Гипсофила",     ["гипсофил", "gypsophila"]),
    ("Фрезия",        ["фрези", "freesia"]),
]


def rub(x, _=None):
    return f"{int(x):,}".replace(",", "\u202f")


def detect_type(name):
    name_low = name.lower()
    for label, keywords in FLOWER_TYPES:
        if any(kw in name_low for kw in keywords):
            return label
    return None


class MarginData:

    buy_source  = "BBFlowers ×15"
    sell_source = "Uflor"

    def __init__(self):
        self.df = None

    def load(self):
        try:
            df_buy  = pd.read_csv("bouquets_data.csv", encoding="utf-8-sig")
            df_sell = pd.read_csv("marketplaces_data.csv", encoding="utf-8-sig")
        except FileNotFoundError as e:
            print(f"! Файл не найден: {e}")
            return False

        df_buy  = df_buy[df_buy["source"]   == self.buy_source].copy()
        df_sell = df_sell[df_sell["source"] == self.sell_source].copy()

        df_buy["flower_type"]  = df_buy["name"].apply(detect_type)
        df_sell["flower_type"] = df_sell["name"].apply(detect_type)

        df_buy  = df_buy.dropna(subset=["flower_type"])
        df_sell = df_sell.dropna(subset=["flower_type"])

        buy_avg  = df_buy.groupby("flower_type")["price"].mean().reset_index()
        sell_avg = df_sell.groupby("flower_type")["price"].mean().reset_index()
        buy_avg.columns  = ["flower_type", "buy_price"]
        sell_avg.columns = ["flower_type", "sell_price"]

        df = pd.merge(buy_avg, sell_avg, on = "flower_type")
        df["margin_abs"] = df["sell_price"] - df["buy_price"]
        df["margin_pct"] = (df["margin_abs"] / df["buy_price"] * 100).round(1)
        df = df.sort_values("margin_abs", ascending=False).reset_index(drop=True)

        self.df = df
        return True


class MarginChart:

    def __init__(self, df):
        self.df = df

    def draw(self):
        df = self.df
        colors_margin = ["#2D6A4F" if v > 0 else "#E76F51" for v in df["margin_abs"]]

        fig, axes = plt.subplots(1, 3, figsize=(20, max(6, len(df) * 0.5)))
        fig.suptitle("Маржинальность по типу цветка\n(закупка BBFlowers vs маркетплейсы)",
                     fontsize=14, fontweight="bold")

        ax = axes[0]
        bars = ax.barh(df["flower_type"], df["margin_abs"],
                       color=colors_margin, edgecolor="white", height=0.65)
        ax.set_xlabel("Маржа, руб. (за 15 шт.)", fontsize=10)
        ax.set_title("Абсолютная маржа", fontsize=12, fontweight="bold")
        ax.axvline(0, color="#333", linewidth=1)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(rub))
        ax.grid(axis="x", alpha=0.25)
        ax.set_axisbelow(True)
        avg_line = df["margin_abs"].mean()
        ax.axvline(avg_line, color="#E76F51", linewidth=1.6, linestyle="--",
                   label=f"Среднее {avg_line:+,.0f} ₽".replace(",", "\u202f"))
        ax.legend(fontsize=9)
        gap = df["margin_abs"].abs().max() * 0.02
        for bar, val in zip(bars, df["margin_abs"]):
            x = val + gap if val >= 0 else val - gap
            ha = "left" if val >= 0 else "right"
            ax.text(x, bar.get_y() + bar.get_height() / 2,
                    f"{val:+,.0f}\u202f₽".replace(",", "\u202f"),
                    va="center", ha=ha, fontsize=9, fontweight="bold")

        ax = axes[1]
        colors_pct = ["#2D6A4F" if v > 0 else "#E76F51" for v in df["margin_pct"]]
        bars = ax.barh(df["flower_type"], df["margin_pct"],
                       color=colors_pct, edgecolor="white", height=0.65)
        ax.set_xlabel("Маржа, %", fontsize=10)
        ax.set_title("Маржа в процентах", fontsize=12, fontweight="bold")
        ax.axvline(0, color="#333", linewidth=1)
        ax.grid(axis="x", alpha=0.25)
        ax.set_axisbelow(True)
        gap_pct = df["margin_pct"].abs().max() * 0.02
        for bar, val in zip(bars, df["margin_pct"]):
            x = val + gap_pct if val >= 0 else val - gap_pct
            ha = "left" if val >= 0 else "right"
            ax.text(x, bar.get_y() + bar.get_height() / 2,
                    f"{val:+.1f}%", va="center", ha=ha, fontsize=9, fontweight="bold")

        ax = axes[2]
        y = range(len(df))
        h = 0.35
        ax.barh([i + h/2 for i in y], df["sell_price"],
                height=h, color="#E76F51", edgecolor="white", label="Розница (маркетплейс)")
        ax.barh([i - h/2 for i in y], df["buy_price"],
                height=h, color="#74C69D", edgecolor="white", label="Закупка (BBFlowers)")
        ax.set_yticks(list(y))
        ax.set_yticklabels(df["flower_type"], fontsize=9)
        ax.set_xlabel("Цена за 15 шт., руб.", fontsize=10)
        ax.set_title("Закупка vs Розница", fontsize=12, fontweight="bold")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(rub))
        ax.grid(axis="x", alpha=0.25)
        ax.set_axisbelow(True)
        ax.legend(fontsize=9)

        plt.tight_layout()
        plt.savefig("05_margin_analysis.png", dpi=150, bbox_inches="tight")
        plt.close()


class MarginPctChart:

    def __init__(self, df):
        self.df = df.sort_values("margin_pct", ascending=True).reset_index(drop=True)

    def draw(self):
        df = self.df
        colors = ["#2D6A4F" if v > 0 else "#E76F51" for v in df["margin_pct"]]

        fig, ax = plt.subplots(figsize=(12, max(5, len(df) * 0.55)))
        fig.suptitle("Маржа в процентах по типу цветка\n(закупка BBFlowers vs маркетплейсы)",
                     fontsize=14, fontweight="bold")

        bars = ax.barh(df["flower_type"], df["margin_pct"],
                       color=colors, edgecolor="white", height=0.65)
        ax.set_xlabel("Маржа, %", fontsize=11)
        ax.axvline(0, color="#333", linewidth=1)
        ax.axvline(df["margin_pct"].mean(), color="#E76F51", linewidth=1.6,
                   linestyle="--", label=f"Среднее {df['margin_pct'].mean():.1f}%")
        ax.grid(axis="x", alpha=0.25)
        ax.set_axisbelow(True)
        ax.legend(fontsize=10)

        gap = df["margin_pct"].abs().max() * 0.02
        for bar, val in zip(bars, df["margin_pct"]):
            x = val + gap if val >= 0 else val - gap
            ha = "left" if val >= 0 else "right"
            ax.text(x, bar.get_y() + bar.get_height() / 2,
                    f"{val:+.1f}%", va="center", ha=ha, fontsize=10, fontweight="bold")

        plt.tight_layout()
        plt.savefig("06_margin_pct.png", dpi=150, bbox_inches="tight")
        plt.close()


class PriceTable:

    def __init__(self, df):
        self.df = df

    def draw(self):
        df = self.df
        fig, ax = plt.subplots(figsize=(12, max(4, len(df) * 0.55)))
        ax.axis("off")
        ax.set_title("Закупочные и розничные цены за 15 цветков",
                     fontsize=14, fontweight="bold", pad=16)

        col_headers = ["Тип цветка", "Закупка ₽\n(BBFlowers)", "Розница ₽\n(маркетплейсы)", "Маржа ₽", "Маржа %"]
        table_data  = []
        for _, row in df.iterrows():
            table_data.append([
                row["flower_type"],
                f"{row['buy_price']:,.0f}".replace(",", "\u202f"),
                f"{row['sell_price']:,.0f}".replace(",", "\u202f"),
                f"{row['margin_abs']:+,.0f}".replace(",", "\u202f"),
                f"{row['margin_pct']:+.1f}%",
            ])

        tbl = ax.table(cellText=table_data, colLabels=col_headers,
                       cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(11)
        tbl.scale(1, 2.2)

        for j in range(len(col_headers)):
            tbl[0, j].set_facecolor("#2D6A4F")
            tbl[0, j].set_text_props(color="white", fontweight="bold")

        for i in range(1, len(table_data) + 1):
            bg = "#f0f9f4" if i % 2 == 0 else "white"
            for j in range(len(col_headers)):
                tbl[i, j].set_facecolor(bg)
            color = "#2D6A4F" if df.iloc[i - 1]["margin_abs"] > 0 else "#E76F51"
            tbl[i, 3].set_text_props(color=color, fontweight="bold")
            tbl[i, 4].set_text_props(color=color, fontweight="bold")

        plt.tight_layout()
        plt.savefig("07_price_table.png", dpi=150, bbox_inches="tight")
        plt.close()


def main():
    data = MarginData()
    if not data.load():
        return

    df = data.df

    if df.empty:
        print("Нет совпадающих типов цветков — проверь данные")
        return

    MarginChart(df).draw()
    MarginPctChart(df).draw()
    PriceTable(df).draw()

    avg_abs = df["margin_abs"].mean()
    avg_pct = df["margin_pct"].mean()

    print("Готово! Файлы:")
    print("05_margin_analysis.png")
    print("06_margin_pct.png")
    print("07_price_table.png")

    print("\nВсе позиции по убыванию маржи:")
    for i, (_, row) in enumerate(df.iterrows(), 1):
        print(f"  {i:2d}. {row['flower_type']:15s}  "
              f"закупка {row['buy_price']:,.0f} ₽  "
              f"розница {row['sell_price']:,.0f} ₽  "
              f"маржа {row['margin_abs']:+,.0f} ₽ ({row['margin_pct']:+.1f}%)")

    print(f"\nСредняя маржинальность:")
    print(f"Абсолютная : {avg_abs:+,.0f} руб. за 15 цветков")
    print(f"В процентах: {avg_pct:+.1f}%")


if __name__ == "__main__":
    main()

Закупка (BBFlowers): 14 позиций
Розница (маркетплейсы): 145 позиций

Типы в закупке:  ['Альстромерия', 'Гербера', 'Пион', 'Роза', 'Хризантема', 'Эустома']
Типы в рознице:  ['Альстромерия', 'Гвоздика', 'Гербера', 'Гортензия', 'Ирис', 'Пион', 'Подсолнух', 'Роза', 'Сирень', 'Тюльпан', 'Хризантема', 'Эустома']

Совпадающих типов: 6
 flower_type  buy_price   sell_price  margin_abs  margin_pct
     Эустома     4500.0 10917.000000 6417.000000       142.6
  Хризантема     2300.0  7419.571429 5119.571429       222.6
     Гербера     1875.0  6492.206897 4617.206897       246.3
        Пион     4500.0  8923.909091 4423.909091        98.3
        Роза     2100.0  5342.542373 3242.542373       154.4
Альстромерия     2250.0  4491.000000 2241.000000        99.6

Готово! Файлы:
  05_margin_analysis.png
  06_margin_pct.png
  07_price_table.png

Все позиции по убыванию маржи:
   1. Эустома          закупка 4,500 ₽  розница 10,917 ₽  маржа +6,417 ₽ (+142.6%)
   2. Хризантема       закупка 2,300 ₽  розниц

In [23]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"

FLOWER_TYPES = [
    ("Роза",          ["роза", "розы", "розов", "розой", "роз ", "rose", "roses"]),
    ("Тюльпан",       ["тюльпан", "tulip"]),
    ("Гербера",       ["гербер", "герберы", "gerbera", "germini", "гермини"]),
    ("Альстромерия",  ["альстромери", "alstromer"]),
    ("Хризантема",    ["хризантем", "chrys", "сантини", "santini"]),
    ("Дендробиум",    ["дендробиум", "dendrobium"]),
    ("Гвоздика",      ["гвоздик", "dianthus", "диантус"]),
    ("Ирис",          ["ирис", "iris"]),
    ("Пион",          ["пион", "peony", "пеони"]),
    ("Лилия",         ["лили", "lili", "lily"]),
    ("Гортензия",     ["гортензи", "hydrangea"]),
    ("Эустома",       ["эустом", "eustom", "lisianthus", "лизиантус"]),
    ("Подсолнух",     ["подсолнух", "sunflower"]),
    ("Ромашка",       ["ромашк", "chamomile"]),
    ("Сирень",        ["сирен", "lilac"]),
    ("Орхидея",       ["орхидея", "орхидей", "orchid"]),
    ("Гипсофила",     ["гипсофил", "gypsophila"]),
    ("Фрезия",        ["фрези", "freesia"]),
]


def detect_type(name):
    name_low = name.lower()
    for label, keywords in FLOWER_TYPES:
        if any(kw in name_low for kw in keywords):
            return label
    return None


def main():
    df_buy  = pd.read_csv("bouquets_data.csv",     encoding="utf-8-sig")
    df_sell = pd.read_csv("marketplaces_data.csv", encoding="utf-8-sig")

    df_buy  = df_buy[df_buy["source"]   == "BBFlowers ×15"].copy()
    df_sell = df_sell[df_sell["source"] == "Dostavka"].copy()

    df_buy["flower_type"]  = df_buy["name"].apply(detect_type)
    df_sell["flower_type"] = df_sell["name"].apply(detect_type)

    buy_avg  = df_buy.groupby("flower_type")["price"].mean().reset_index()
    sell_avg = df_sell.groupby("flower_type")["price"].mean().reset_index()
    buy_avg.columns  = ["flower_type", "buy_price"]
    sell_avg.columns = ["flower_type", "sell_price"]

    df = pd.merge(buy_avg, sell_avg, on="flower_type")
    df["margin_abs"] = df["sell_price"] - df["buy_price"]
    df["margin_pct"] = (df["margin_abs"] / df["buy_price"] * 100).round(1)
    df = df.sort_values("margin_abs", ascending=False).reset_index(drop=True)

    # Таблица
    fig, ax = plt.subplots(figsize=(12, max(4, len(df) * 0.6)))
    ax.axis("off")
    ax.set_title("Закупочные и розничные цены за 15 цветков\n(BBFlowers vs Dostavka)",
                 fontsize=14, fontweight="bold", pad=16)

    col_headers = ["Тип цветка", "Закупка ₽\n(BBFlowers)", "Розница ₽\n(Uflor)", "Маржа ₽", "Маржа %"]
    table_data = []
    for _, row in df.iterrows():
        table_data.append([
            row["flower_type"],
            f"{row['buy_price']:,.0f}".replace(",", "\u202f"),
            f"{row['sell_price']:,.0f}".replace(",", "\u202f"),
            f"{row['margin_abs']:+,.0f}".replace(",", "\u202f"),
            f"{row['margin_pct']:+.1f}%",
        ])

    tbl = ax.table(
        cellText=table_data,
        colLabels=col_headers,
        cellLoc="center",
        loc="center",
        bbox=[0, 0, 1, 1],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1, 2.2)

    for j in range(len(col_headers)):
        tbl[0, j].set_facecolor("#2D6A4F")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    for i in range(1, len(table_data) + 1):
        bg = "#f0f9f4" if i % 2 == 0 else "white"
        for j in range(len(col_headers)):
            tbl[i, j].set_facecolor(bg)
        margin_val = df.iloc[i - 1]["margin_abs"]
        color = "#2D6A4F" if margin_val > 0 else "#E76F51"
        tbl[i, 3].set_text_props(color=color, fontweight="bold")
        tbl[i, 4].set_text_props(color=color, fontweight="bold")

    plt.tight_layout()
    plt.savefig("price_table.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Готово: price_table.png")


if __name__ == "__main__":
    main()

Готово: price_table.png
